#### **Read a few examples**

In [7]:
import os
import json
import torch
from itertools import islice

path = "/workspace/sam/minireason_archives/data_pts/colton_h3/countdown_colton_h3_full"

def iter_dataset(path):
    if os.path.isdir(path):
        manifest = os.path.join(path, "manifest.json")
        if os.path.exists(manifest):
            with open(manifest) as f:
                names = json.load(f)["shards"]
            paths = [os.path.join(path, name) for name in names]
        else:
            paths = sorted(
                os.path.join(path, fn)
                for fn in os.listdir(path)
                if fn.startswith("shard_") and fn.endswith(".pt")
            )
        for p in paths:
            for record in torch.load(p, weights_only=False):
                yield record
    else:
        for record in torch.load(path, weights_only=False):
            yield record

for i, r in enumerate(islice(iter_dataset(path), 10)):
    print(f"\n--- sample {i} ---")
    print(r)


--- sample 0 ---
{'target': 100, 'answer': 100, 'gap': 0, 'inputs': (1, 8, 8, 9, 9, 100), 'expr': '100', 'num_used': 1, 'num_ops': 0, 'max_interm': 0, 'explanation': [], 'score': 10}

--- sample 1 ---
{'target': 101, 'answer': 101, 'gap': 0, 'inputs': (1, 8, 8, 9, 9, 100), 'expr': '(100+1)', 'num_used': 2, 'num_ops': 1, 'max_interm': 100, 'explanation': ['100+1=101'], 'score': 10}

--- sample 2 ---
{'target': 102, 'answer': 102, 'gap': 0, 'inputs': (1, 8, 8, 9, 9, 100), 'expr': '(100+(9-(8-1)))', 'num_used': 4, 'num_ops': 2, 'max_interm': 100, 'explanation': ['8-1=7', '9-7=2', '100+2=102'], 'score': 10}

--- sample 3 ---
{'target': 103, 'answer': 103, 'gap': 0, 'inputs': (1, 8, 8, 9, 9, 100), 'expr': '(100+(9-((8-(9-8))-1)))', 'num_used': 6, 'num_ops': 2, 'max_interm': 100, 'explanation': ['9-8=1', '8-1=7', '7-1=6', '9-6=3', '100+3=103'], 'score': 10}

--- sample 4 ---
{'target': 104, 'answer': 104, 'gap': 0, 'inputs': (1, 8, 8, 9, 9, 100), 'expr': '(100+(8/(9-(8-1))))', 'num_used': 5

#### **Deeper Check with output distributions**

In [ ]:
import os
import json
import ast
import torch
import random
from collections import Counter
from tqdm.auto import tqdm
from pprint import pprint

path = "/workspace/sam/minireason_archives/data_pts/colton_h3/countdown_colton_h3_full"
RANDOM_EXAMPLES = 5
SEED = 42

random.seed(SEED)

def shard_paths(path):
    if os.path.isdir(path):
        manifest = os.path.join(path, "manifest.json")
        if os.path.exists(manifest):
            with open(manifest) as f:
                names = json.load(f)["shards"]
            return [os.path.join(path, name) for name in names]
        return sorted(
            os.path.join(path, fn)
            for fn in os.listdir(path)
            if fn.startswith("shard_") and fn.endswith(".pt")
        )
    return [path]

def iter_dataset(path):
    for p in shard_paths(path):
        data = torch.load(p, weights_only=False, map_location="cpu",)
        for record in data:
            yield record

def eval_expr_and_used(expr_str):
    tree = ast.parse(expr_str, mode="eval")

    def rec(node):
        if isinstance(node, ast.Expression):
            return rec(node.body)

        if isinstance(node, ast.Constant):
            if not isinstance(node.value, int):
                raise ValueError(f"Non-integer constant: {node.value}")
            return node.value, Counter([node.value])

        if isinstance(node, ast.BinOp):
            lv, lused = rec(node.left)
            rv, rused = rec(node.right)
            used = lused + rused

            if isinstance(node.op, ast.Add):
                return lv + rv, used

            if isinstance(node.op, ast.Sub):
                if lv <= rv:
                    raise ValueError(f"Illegal subtraction: {lv} - {rv}")
                return lv - rv, used

            if isinstance(node.op, ast.Mult):
                return lv * rv, used

            if isinstance(node.op, ast.Div):
                if rv == 0:
                    raise ValueError("Division by zero")
                if lv % rv != 0:
                    raise ValueError(f"Non-integer division: {lv} / {rv}")
                return lv // rv, used

            raise ValueError(f"Unsupported operator: {ast.dump(node.op)}")

        raise ValueError(f"Unsupported syntax: {ast.dump(node)}")

    return rec(tree)

def validate_record(r):
    expr = r["expr"]
    answer = r["answer"]
    inputs = r["inputs"]

    if expr is None:
        return (answer is None), "expr is None"

    try:
        value, used = eval_expr_and_used(expr)
    except Exception as e:
        return False, f"parse/eval error: {e}"

    available = Counter(inputs)
    for k, v in used.items():
        if available[k] < v:
            return False, f"used value {k} {v} times but only {available[k]} available"

    if value != answer:
        return False, f"expr evaluates to {value}, but answer is {answer}"

    return True, "ok"

def pct(x, n):
    return 100.0 * x / n if n else 0.0

def split_trace_lines(trace: str):
    parts = [part.strip() for part in trace.split(";") if part.strip()]
    lines = []
    for part in parts:
        if part.startswith("EXPR"):
            lines.append(part)
        else:
            lines.append(part + " ;")
    return lines

paths = shard_paths(path)
total = len(paths) * 900 if os.path.isdir(path) else None

score_counts = Counter()
gap_counts = Counter()
num_used_counts = Counter()
bad = []

sample_pool = []
checked = 0
total_score = 0
difficult = 0

with tqdm(total=total, desc="Checking samples", unit="sample") as pbar:
    for r in iter_dataset(path):
        ok, msg = validate_record(r)

        if not ok:
            bad.append({
                "target": r["target"],
                "answer": r["answer"],
                "inputs": r["inputs"],
                "expr": r["expr"],
                "reason": msg,
            })

        sc = r["score"]
        gap = r["gap"]
        nu = r["num_used"]

        score_counts[sc] += 1
        gap_counts[gap] += 1
        if nu is not None:
            num_used_counts[nu] += 1
        total_score += sc

        if (
            sc == 10
            and nu == 6
            and r["max_interm"] is not None
            and r["max_interm"] >= 1000
            and r["num_ops"] == 4
        ):
            difficult += 1

        checked += 1
        pbar.update(1)

        if checked % 100000 == 0:
            pbar.set_postfix_str(
                f"target={r['target']} score={r['score']} gap={r['gap']} inputs={r['inputs']}",
                refresh=False,
            )

        # reservoir sampling for random examples
        if len(sample_pool) < RANDOM_EXAMPLES:
            sample_pool.append(r)
        else:
            j = random.randint(1, checked)
            if j <= RANDOM_EXAMPLES:
                sample_pool[j - 1] = r

print(f"\nChecked {checked:,} samples")
print(f"Bad samples: {len(bad):,}")

print("\n=== Point distribution ===")
for sc in [10, 7, 5, 0]:
    c = score_counts[sc]
    print(f"Score {sc:>2}: {c:>10,}  ({pct(c, checked):6.2f}%)")

print(f"\nExpected score: {total_score / checked:.4f}")

print("\n=== Gap distribution ===")
for g in sorted(k for k in gap_counts if k is not None)[:15]:
    print(f"gap={g:>2}: {gap_counts[g]:>10,}")
if None in gap_counts:
    print(f"gap=None: {gap_counts[None]:>10,}")

print("\n=== num_used distribution ===")
for nu in sorted(num_used_counts):
    c = num_used_counts[nu]
    print(f"num_used={nu}: {c:>10,}  ({pct(c, checked):6.2f}%)")

easy = sum(c for nu, c in num_used_counts.items() if nu <= 3)
medium = num_used_counts[4]
hard = sum(c for nu, c in num_used_counts.items() if nu >= 5)

print("\n=== Difficulty buckets ===")
print(f"Easy   (U<=3): {easy:>10,}  ({pct(easy, checked):6.2f}%)")
print(f"Medium (U=4) : {medium:>10,}  ({pct(medium, checked):6.2f}%)")
print(f"Hard   (U>=5): {hard:>10,}  ({pct(hard, checked):6.2f}%)")
print(f"Difficult    : {difficult:>10,}  ({pct(difficult, checked):6.2f}%)")

print("\n=== 5 random examples ===")
for i, r in enumerate(sample_pool, 1):
    print(f"\n--- random sample {i} ---")
    pprint({
        "target": r["target"],
        "answer": r["answer"],
        "gap": r["gap"],
        "score": r["score"],
        "inputs": r["inputs"],
        "expr": r["expr"],
        "num_used": r["num_used"],
        "num_ops": r["num_ops"],
        "max_interm": r["max_interm"],
        "explanation": r["explanation"],
    })

if bad:
    print("\n=== First few bad samples ===")
    for x in bad[:10]:
        pprint(x)
else:
    print("\nAll checked expressions matched their stored answers.")

Checking samples:   0%|                                | 901/11918700 [00:00<25:15, 7863.80sample/s, target=999 score=10 gap=0 inputs=(1, 8, 8, 9, 9, 100)]

Checking samples:   8%|██▏                        | 987598/11918700 [01:00<11:06, 16407.88sample/s, target=389 score=10 gap=0 inputs=(1, 2, 6, 9, 10, 100)]


KeyboardInterrupt: 